> **Optional follow-up: object distance fusion.** Start with the three comparison notebooks first. This older exploratory notebook needs separately exported detector predictions, metric depth maps, and object-distance labels. It is not part of the shared-model benchmark.

# Object-to-distance fusion evaluation

Object detection and depth should be integrated for the driving-assistance behavior, but their model benchmarks remain separate. This notebook evaluates the fused result: detected object box -> robust depth sample -> object distance estimate.

A depth map alone does not identify which distance belongs to a vehicle or pedestrian. A box alone does not provide distance. The fusion step connects the two.

## Data contract

Create `data/fusion/manifest.csv` with these columns:

`image,depth,predictions,ground_truth`

- `depth`: a `.npy` depth map in meters, with zero for invalid pixels.
- `predictions`: JSON list of detector outputs with `class`, `box` `[x1,y1,x2,y2]`, and optional `score`.
- `ground_truth`: JSON list with the same box/class fields and a `distance_m` value from projected LiDAR or 3D annotations.

The predictions can come from any object notebook model, and the depth map can come from any depth notebook model. This makes the fusion notebook reusable across model combinations.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

ROOT = Path.cwd()
if not (ROOT / 'configs').exists():
    ROOT = Path.cwd().parent
MANIFEST = ROOT / 'data/fusion/manifest.csv'
REPORT_DIR = ROOT / 'reports/fusion'

def load_manifest(path):
    if not path.is_file():
        print(f'No fusion manifest found at {MANIFEST}. Export detector boxes, depth maps, and object distance labels first.')
        return pd.DataFrame()
    frame = pd.read_csv(path)
    required = {'image', 'depth', 'predictions', 'ground_truth'}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f'Manifest is missing columns: {sorted(missing)}')
    return frame

manifest = load_manifest(MANIFEST)
print(f'Fusion frames: {len(manifest)}')

In [ ]:
def robust_box_depth(depth, box, inner_fraction=0.5):
    height, width = depth.shape[:2]
    x1, y1, x2, y2 = map(float, box)
    center_x, center_y = (x1 + x2) / 2, (y1 + y2) / 2
    half_width, half_height = (x2 - x1) * inner_fraction / 2, (y2 - y1) * inner_fraction / 2
    left, right = max(0, int(center_x - half_width)), min(width, int(center_x + half_width))
    top, bottom = max(0, int(center_y - half_height)), min(height, int(center_y + half_height))
    values = depth[top:bottom, left:right]
    valid = values[np.isfinite(values) & (values > 0)]
    return float(np.median(valid)) if valid.size else float('nan')

def estimate_object_distances(depth, predictions):
    estimates = []
    for prediction in predictions:
        estimate = robust_box_depth(depth, prediction['box'])
        estimates.append({**prediction, 'distance_m': estimate})
    return estimates

def match_objects(predictions, ground_truth, max_center_error_px=80):
    matches = []
    unused = set(range(len(ground_truth)))
    for prediction in predictions:
        candidates = [index for index in unused if ground_truth[index].get('class') == prediction.get('class')]
        if not candidates:
            continue
        px = (prediction['box'][0] + prediction['box'][2]) / 2
        py = (prediction['box'][1] + prediction['box'][3]) / 2
        index = min(candidates, key=lambda item: ((ground_truth[item]['box'][0] + ground_truth[item]['box'][2]) / 2 - px) ** 2 + ((ground_truth[item]['box'][1] + ground_truth[item]['box'][3]) / 2 - py) ** 2)
        gx = (ground_truth[index]['box'][0] + ground_truth[index]['box'][2]) / 2
        gy = (ground_truth[index]['box'][1] + ground_truth[index]['box'][3]) / 2
        if (gx - px) ** 2 + (gy - py) ** 2 <= max_center_error_px ** 2:
            matches.append((prediction, ground_truth[index]))
            unused.remove(index)
    return matches

def evaluate_frame(row):
    depth = np.load((MANIFEST.parent / row.depth).resolve()).astype(np.float32)
    predictions = estimate_object_distances(depth, json.loads(row.predictions))
    ground_truth = json.loads(row.ground_truth)
    matches = match_objects(predictions, ground_truth)
    errors = [prediction['distance_m'] - actual['distance_m'] for prediction, actual in matches if np.isfinite(prediction['distance_m'])]
    return {'frames': 1, 'matched_objects': len(matches), 'distance_mae_m': float(np.mean(np.abs(errors))) if errors else np.nan, 'distance_rmse_m': float(np.sqrt(np.mean(np.square(errors)))) if errors else np.nan, 'within_1m': float(np.mean(np.abs(errors) <= 1.0)) if errors else np.nan}

if manifest.empty:
    results = pd.DataFrame([{'frames': 0, 'matched_objects': 0, 'distance_mae_m': np.nan, 'distance_rmse_m': np.nan, 'within_1m': np.nan, 'status': 'not_ready', 'reason': 'Add data/fusion/manifest.csv with RGB images, depth maps, detector predictions, and distance labels.'}])
    print('No labeled fusion data found. The table below is a readiness report, not an accuracy result.')
else:
    frame_results = pd.DataFrame([evaluate_frame(row) for row in manifest.itertuples(index=False)])
    results = frame_results.mean(numeric_only=True).to_frame().T
    print(results.to_string(index=False))
    display(results)
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    results.to_csv(REPORT_DIR / 'object_distance_results.csv', index=False)

## Visual fusion check

The green box is the detector output. Its distance estimate is the median valid depth in the inner part of the box, which reduces contamination from the road and background. The red label is the ground-truth object distance.

In [ ]:
def show_fusion_frame(row):
    image = np.asarray(Image.open((MANIFEST.parent / row.image).resolve()).convert('RGB'))
    depth = np.load((MANIFEST.parent / row.depth).resolve()).astype(np.float32)
    predictions = estimate_object_distances(depth, json.loads(row.predictions))
    actual = json.loads(row.ground_truth)
    figure, axis = plt.subplots(figsize=(12, 7))
    axis.imshow(image)
    for prediction in predictions:
        x1, y1, x2, y2 = prediction['box']
        axis.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor='lime', linewidth=2))
        axis.text(x1, y1, f"{prediction.get('class', 'object')}: {prediction['distance_m']:.1f}m", color='black', backgroundcolor='lime')
    for label in actual:
        x1, y1, x2, y2 = label['box']
        axis.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor='red', linewidth=1, linestyle='--'))
        axis.text(x1, y2, f"actual: {label['distance_m']:.1f}m", color='white', backgroundcolor='red')
    axis.axis('off'); axis.set_title('Green = predicted object distance; red dashed = actual distance'); plt.show()

if manifest.empty:
    print('Visual fusion examples will appear after a manifest is configured.')
else:
    show_fusion_frame(manifest.iloc[0])

## Interpretation boundary

This notebook evaluates object-to-distance association. It does not yet evaluate tracking, camera calibration error, temporal smoothing, uncertainty, time-to-collision, or driver-warning timing. Those are required before using an estimate to trigger a safety alert.